# Stage 6: Connections Between Research Directions

We want to investigate whether related NLP research directions could suggest potential areas worth exploring for academic cooperation.

## 1. Finding research topics with high similarity

In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Load existing artifacts
artifacts = Path("artifacts")
if not artifacts.exists():
    artifacts = Path("../artifacts")

kmeans = joblib.load(artifacts / "final_kmeans_160.joblib")
profiles = pd.read_csv(
    artifacts / "final_cluster_profiles.csv"
).set_index("cluster_id")

# Compute pairwise cosine similarity between cluster centroids
similarity = cosine_similarity(kmeans.cluster_centers_)

# Select the 12 most similar unique pairs (excluding self-pairs)
i, j = np.triu_indices_from(similarity, k=1)
top_idx = np.argsort(similarity[i, j])[-12:][::-1]

# Display similarity and characteristic keywords
pairs = pd.DataFrame([
    {
        "Cluster A": int(i[t]),
        "Cluster B": int(j[t]),
        "Cosine similarity": round(float(similarity[i[t], j[t]]), 3),
        "Keywords A": profiles.loc[i[t], "top_terms"],
        "Keywords B": profiles.loc[j[t], "top_terms"]
    }
    for t in top_idx
])

with pd.option_context("display.max_colwidth", None):
    display(pairs)


,Cluster A,Cluster B,Cosine similarity,Keywords A,Keywords B
0,147,149,0.886,"reasoning, logical, llms, logical reasoning, models, language, logic, symbolic, commonsense, language models, tasks, large language","reasoning, models, reasoning models, thinking, llms, lrms, step, model, tasks, large, capabilities, accuracy"
1,55,149,0.885,"reasoning, cot, thought, chain, chain thought, thought cot, models, llms, step, cot reasoning, steps, language","reasoning, models, reasoning models, thinking, llms, lrms, step, model, tasks, large, capabilities, accuracy"
2,32,55,0.864,"cot, reasoning, thought, chain, chain thought, prompting, thought cot, llms, cot prompting, models, cot reasoning, step","reasoning, cot, thought, chain, chain thought, thought cot, models, llms, step, cot reasoning, steps, language"
3,11,128,0.843,"medical, clinical, patient, llms, healthcare, diagnosis, diagnostic, models, reasoning, patients, language, large","medical, llms, models, healthcare, domain, knowledge, medical knowledge, language, medical domain, large, clinical, med"
4,22,72,0.839,"asr, speech, speech recognition, recognition, automatic speech, recognition asr, automatic, asr systems, wer, error, model, asr models","speech, speech recognition, recognition, asr, automatic speech, automatic, model, data, recognition asr, supervised, self supervised, training"
5,72,140,0.817,"speech, speech recognition, recognition, asr, automatic speech, automatic, model, data, recognition asr, supervised, self supervised, training","speech, tts, text, text speech, synthesis, model, speech synthesis, models, speech text, prosody, spoken, audio"
6,55,147,0.792,"reasoning, cot, thought, chain, chain thought, thought cot, models, llms, step, cot reasoning, steps, language","reasoning, logical, llms, logical reasoning, models, language, logic, symbolic, commonsense, language models, tasks, large language"
7,5,148,0.791,"dialogue, response, persona, responses, generation, response generation, dialogue generation, empathetic, knowledge, human, conversation, model","dialogue, dialogue systems, task oriented, oriented, systems, oriented dialogue, task, dialogues, dialogue state, dst, tod, state tracking"
8,11,132,0.789,"medical, clinical, patient, llms, healthcare, diagnosis, diagnostic, models, reasoning, patients, language, large","clinical, notes, patient, clinical notes, health, ehr, patients, data, medical, models, electronic, electronic health"
9,140,154,0.729,"speech, tts, text, text speech, synthesis, model, speech synthesis, models, speech text, prosody, spoken, audio","speech, translation, speech translation, st, end, simultaneous, speech text, speech speech, text, simultaneous speech, text translation, end end"


## 2. Find the nearest research directions

In [2]:
# Identify research directions closest to the three emerging niche topics

niche_topics = {
    74: "KV-cache optimization",
    107: "Machine unlearning",
    112: "Cultural alignment"
}

# Reuse the similarity matrix and profiles from Step 6.1
results = []

for cluster_id, topic in niche_topics.items():
    neighbors = np.argsort(similarity[cluster_id])[::-1]
    neighbors = neighbors[neighbors != cluster_id][:4]

    for neighbor in neighbors:
        results.append({
            "Niche topic": f"{topic} ({cluster_id})",
            "Related cluster": int(neighbor),
            "Cosine similarity": round(
                float(similarity[cluster_id, neighbor]), 3
            ),
            "Related cluster keywords": profiles.loc[
                neighbor, "top_terms"
            ]
        })

niche_connections = pd.DataFrame(results)

with pd.option_context("display.max_colwidth", None):
    display(niche_connections)


,Niche topic,Related cluster,Cosine similarity,Related cluster keywords
0,KV-cache optimization (74),85,0.375,"inference, model, models, time, llms, llm, efficiency, latency, language, performance, efficient, cost"
1,KV-cache optimization (74),118,0.374,"compression, model compression, prompt compression, model, compressed, compression methods, models, context, language, llms, performance, compression techniques"
2,KV-cache optimization (74),26,0.324,"attention, transformer, self attention, heads, transformers, attention mechanism, mechanism, self, head, long, linear, sparse"
3,KV-cache optimization (74),54,0.317,"long, long context, context, length, llms, models, attention, contexts, tasks, context length, tokens, retrieval"
4,Machine unlearning (107),4,0.287,"models, model, language, language models, training, large, language model, pretraining, large language, data, performance, tasks"
5,Machine unlearning (107),76,0.259,"llms, models, language, large language, language models, large, models llms, llm, tasks, performance, human, capabilities"
6,Machine unlearning (107),25,0.245,"llm, llms, llm based, large language, large, based, language, models, judge, model, language models, framework"
7,Machine unlearning (107),21,0.241,"knowledge, commonsense, llms, models, language, external, language models, external knowledge, factual, commonsense knowledge, reasoning, model"
8,Cultural alignment (112),76,0.394,"llms, models, language, large language, language models, large, models llms, llm, tasks, performance, human, capabilities"
9,Cultural alignment (112),14,0.357,"evaluation, llms, benchmark, benchmarks, models, llm, language, large language, contamination, large, language models, model"
